# Advanced RAG 101

## One question, the right retrieval engine

This notebook builds a small but realistic retrieval system for **Northstar Electronics**. We will use:

- **Vector RAG** for policy questions
- **Graph RAG** for connected facts
- **Text-to-SQL** for exact calculations
- **Exa web search** for fresh external information
- **Corrective RAG (CRAG)** to check weak evidence

The goal is not deep framework knowledge. The goal is knowing **why** each approach exists and **when** to use it.

> Run the notebook from top to bottom. The five sample PDFs are already included in `output/pdf/`.

In [1]:
# Run once. Restart the kernel if Jupyter asks you to.
# Removing legacy brotlipy avoids a known conflict with modern HTTP clients.
%pip uninstall -y -q brotlipy
%pip install -q --upgrade --force-reinstall brotli
%pip install -q --upgrade openai exa-py pypdf aiohttp
%pip install -q numpy pandas networkx python-dotenv

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, sqlite3
from pathlib import Path
from getpass import getpass

import networkx as nx
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from exa_py import Exa
from openai import OpenAI
from pypdf import PdfReader

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
EXA_API_KEY = os.getenv("EXA_API_KEY") or getpass("Exa API key: ")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
PDF_FOLDER = Path("output/pdf")

openai_client = OpenAI(api_key=OPENAI_API_KEY)
exa_client = Exa(api_key=EXA_API_KEY)

print(f"Ready: {CHAT_MODEL} + {EMBEDDING_MODEL}")
print(f"PDF folder: {PDF_FOLDER.resolve()}")

Ready: gpt-5-mini + text-embedding-3-small
PDF folder: C:\Users\Keerthivasan\Documents\ChatGPT\MRP 101\output\pdf


## 1. Our believable document collection

The collection contains about 38 pages across policies, vendor records, a project brief, an operations report, and an incident review. Some facts appear in several documents, and one report deliberately retains **outdated policy values** for audit history.

That mix lets us demonstrate an important lesson: retrieving related words is not the same as retrieving the right evidence.

In [3]:
def chunk_text(text, size=900, overlap=120):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

records = []
for pdf_path in sorted(PDF_FOLDER.glob("*.pdf")):
    reader = PdfReader(pdf_path)
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        for chunk_number, chunk in enumerate(chunk_text(text), start=1):
            records.append({
                "source": pdf_path.name,
                "page": page_number,
                "chunk": chunk_number,
                "text": chunk,
            })

print(f"Loaded {len(list(PDF_FOLDER.glob('*.pdf')))} PDFs")
print(f"Created {len(records)} searchable chunks")
pd.DataFrame(records)[["source", "page", "chunk"]].head()

Loaded 5 PDFs
Created 38 searchable chunks


,source,page,chunk
0,01_customer_returns_policy.pdf,1,1
1,01_customer_returns_policy.pdf,2,1
2,01_customer_returns_policy.pdf,3,1
3,01_customer_returns_policy.pdf,4,1
4,01_customer_returns_policy.pdf,5,1


In [4]:
def embed(texts):
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    matrix = np.array([item.embedding for item in response.data])
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

document_vectors = embed([item["text"] for item in records])

def retrieve(question, top_k=4):
    query_vector = embed([question])[0]
    scores = document_vectors @ query_vector
    best = np.argsort(scores)[::-1][:top_k]
    return [{**records[i], "score": float(scores[i])} for i in best]

def answer_documents(question, top_k=4):
    hits = retrieve(question, top_k)
    context = "\n\n".join(
        f"[{h['source']}, page {h['page']}]\n{h['text']}" for h in hits
    )
    response = openai_client.responses.create(
        model=CHAT_MODEL,
        instructions="Answer only from the evidence. Cite filename and page number. If evidence is missing, say so.",
        input=f"Question: {question}\n\nEvidence:\n{context}",
    )
    print("Retrieved:")
    for h in hits:
        print(f"- {h['source']} p.{h['page']} | score {h['score']:.3f}")
    print("\nAnswer:\n", response.output_text)
    return response.output_text

answer_documents("What is the return window for a damaged electronic product?")

Retrieved:
- 01_customer_returns_policy.pdf p.6 | score 0.552
- 01_customer_returns_policy.pdf p.3 | score 0.549
- 01_customer_returns_policy.pdf p.4 | score 0.540
- 01_customer_returns_policy.pdf p.2 | score 0.507

Answer:
 A damaged product may be returned within 30 calendar days after delivery [01_customer_returns_policy.pdf, page 4].


'A damaged product may be returned within 30 calendar days after delivery [01_customer_returns_policy.pdf, page 4].'

## 2. Where vectors struggle: relationships

Vector search is excellent at finding similar passages. It is weaker when the answer depends on a chain spread across documents:

> How is **Helios Components** connected to the delay in **Project Atlas**?

Graph RAG represents important entities as nodes and their relationships as edges. In production, these relationships can be extracted automatically. Here we write a small, visible graph so the idea stays understandable.

In [5]:
relationships = [
    ("Helios Components", "Orion Control Module", "manufactures"),
    ("Orion Control Module", "Kestrel Logistics", "integrated by"),
    ("Kestrel Logistics", "Project Atlas", "builds cabinets for"),
    ("Quality Incident Q3-17", "Helios Components", "affected a production lot from"),
    ("Quality Incident Q3-17", "Project Atlas", "delayed testing for"),
    ("Priya Nair", "Project Atlas", "leads"),
]

graph = nx.Graph()
for left, right, relation in relationships:
    graph.add_edge(left, right, relation=relation)

path = nx.shortest_path(graph, "Helios Components", "Project Atlas")
path_facts = []
for left, right in zip(path, path[1:]):
    path_facts.append(f"{left} --{graph[left][right]['relation']}--> {right}")

print("\n".join(path_facts))

Helios Components --affected a production lot from--> Quality Incident Q3-17
Quality Incident Q3-17 --delayed testing for--> Project Atlas


In [6]:
def answer_graph(question):
    graph_context = "\n".join(path_facts)
    response = openai_client.responses.create(
        model=CHAT_MODEL,
        instructions="Explain the relationship using only the supplied graph path. Keep the answer under 100 words.",
        input=f"Question: {question}\n\nGraph path:\n{graph_context}",
    )
    print(response.output_text)
    return response.output_text

answer_graph("How is Helios Components connected to the delay in Project Atlas?")

Helios Components affected a production lot that is the subject of Quality Incident Q3-17; Quality Incident Q3-17 then delayed testing for Project Atlas. Thus Helios Components is linked to Project Atlas’s testing delay via that quality incident.


'Helios Components affected a production lot that is the subject of Quality Incident Q3-17; Quality Incident Q3-17 then delayed testing for Project Atlas. Thus Helios Components is linked to Project Atlas’s testing delay via that quality incident.'

## 3. Text-to-SQL: use the database for exact answers

Embeddings can find a paragraph about revenue, but they should not add dozens of numbers. SQL is designed for exact filtering, grouping, and calculation.

We will create a temporary SQLite database **inside this notebook**. It disappears when the kernel stops, so there is no separate database file.

In [13]:
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE revenue (quarter TEXT, region TEXT, category TEXT, revenue_usd INTEGER)")

rows = [
    ("Q2", "North", "Computing", 1200000), ("Q2", "North", "Home Automation", 800000), ("Q2", "North", "Accessories", 500000),
    ("Q3", "North", "Computing", 1300000), ("Q3", "North", "Home Automation", 850000), ("Q3", "North", "Accessories", 520000),
    ("Q2", "South", "Computing", 1000000), ("Q2", "South", "Home Automation", 700000), ("Q2", "South", "Accessories", 450000),
    ("Q3", "South", "Computing", 1250000), ("Q3", "South", "Home Automation", 850000), ("Q3", "South", "Accessories", 600000),
    ("Q2", "East", "Computing", 1100000), ("Q2", "East", "Home Automation", 900000), ("Q2", "East", "Accessories", 550000),
    ("Q3", "East", "Computing", 1170000), ("Q3", "East", "Home Automation", 980000), ("Q3", "East", "Accessories", 590000),
    ("Q2", "West", "Computing", 1300000), ("Q2", "West", "Home Automation", 750000), ("Q2", "West", "Accessories", 600000),
    ("Q3", "West", "Computing", 1380000), ("Q3", "West", "Home Automation", 800000), ("Q3", "West", "Accessories", 650000),
]
connection.executemany("INSERT INTO revenue VALUES (?, ?, ?, ?)", rows)
connection.commit()

pd.read_sql_query("""
    SELECT quarter, region, SUM(revenue_usd) AS total_revenue
    FROM revenue
    GROUP BY quarter, region
    ORDER BY region, quarter
""", connection)

,quarter,region,total_revenue
0,Q2,East,2550000
1,Q3,East,2740000
2,Q2,North,2500000
3,Q3,North,2670000
4,Q2,South,2150000
5,Q3,South,2700000
6,Q2,West,2650000
7,Q3,West,2830000


In [8]:
def ask_database(question):
    schema = "revenue(quarter TEXT, region TEXT, category TEXT, revenue_usd INTEGER)"
    response = openai_client.responses.create(
        model=CHAT_MODEL,
        instructions="Return one SQLite SELECT query only. No markdown and no explanation.",
        input=f"Schema: {schema}\nQuestion: {question}",
    )
    sql = response.output_text.strip().replace("```sql", "").replace("```", "").strip()

    blocked = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "ATTACH", "PRAGMA")
    sql_body = sql.rstrip().rstrip(";")
    starts_read_only = re.match(r"^\s*(SELECT|WITH)\b", sql_body, re.IGNORECASE)
    has_blocked_word = any(re.search(rf"\b{word}\b", sql_body, re.IGNORECASE) for word in blocked)
    if not starts_read_only or has_blocked_word or ";" in sql_body:
        raise ValueError(f"Unsafe SQL rejected: {sql}")

    print("Generated SQL:\n", sql)
    result = pd.read_sql_query(sql, connection)
    display(result)
    return result

ask_database("Which region had the largest revenue increase from Q2 to Q3?")

Generated SQL:
 SELECT region, SUM(CASE WHEN quarter='Q3' THEN revenue_usd ELSE 0 END) - SUM(CASE WHEN quarter='Q2' THEN revenue_usd ELSE 0 END) AS increase FROM revenue GROUP BY region ORDER BY increase DESC LIMIT 1;


,region,increase
0,South,550000


,region,increase
0,South,550000


## 4. Web RAG with Exa - and where Vision RAG fits

Internal PDFs become stale. For questions about recent external events, we use **Exa Search** to retrieve current web evidence, then ask OpenAI to answer from those highlights.

Exa currently advertises **$20 in credits at signup and $10 of free credits each month** on its Starter plan. Pricing can change, so check [exa.ai/pricing](https://exa.ai/pricing).

### Vision RAG, concept only

PDF text extraction can lose charts, diagrams, colors, and layout. Vision RAG retrieves the relevant page as an image and gives it to a vision-capable model. We are not implementing it here because it introduces another ingestion path and would make this beginner lesson too long.

In [10]:
def search_web(question):
    search = exa_client.search(
        question,
        type="auto",
        contents={"highlights": True},
    )
    sources = search.results[:5]
    evidence = []

    print("Exa sources:")
    for number, item in enumerate(sources, start=1):
        highlights = getattr(item, "highlights", None) or []
        if isinstance(highlights, str):
            highlights = [highlights]
        excerpt = " ".join(highlights)
        evidence.append(f"Source {number}: {item.title}\nURL: {item.url}\nEvidence: {excerpt}")
        print(f"{number}. {item.title}\n   {item.url}")

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        instructions="Answer only from the supplied web evidence. Cite source numbers, include the URLs used, and stay under 180 words.",
        input=f"Question: {question}\n\n" + "\n\n".join(evidence),
    )
    print("\nAnswer:\n", response.output_text)
    return response.output_text

search_web("What recent developments are affecting electronics supply chains?")

Exa sources:
1. NOR Flash and SLC NAND production are under threat as capacity gets routed to more profitable products — 'severe undersupply' threatens everyday electronics | Tom's Hardware
   https://www.tomshardware.com/pc-components/dram/nor-flash-and-slc-nand-production-are-under-threat-as-capacity-gets-routed-to-more-profitable-products-severe-undersupply-threatens-everyday-electronics
2. AI boom gives Samsung pricing power as MLCC supply tightens - The Korea Herald
   https://www.koreaherald.com/article/10877230
3. Companies left China to dodge tariffs. Now some are heading back | Awani International
   https://international.astroawani.com/global-news/companies-left-china-dodge-tariffs-now-some-are-heading-back
4. CBP Crackdown And China Curbs Hit US Supply Chains
   https://bigev.news/cbp-crackdown-and-china-curbs-hit-us-supply-chains/
5. Japanese Manufacturers Face Critical Rare-Earth Shortages as China Tightens Export Controls - Beijing Post
   https://beijingpost.com/japanese

'Key developments disrupting electronics supply chains:\n\n- Memory crunch: AI-driven demand is diverting wafer capacity to HBM/DRAM and advanced NAND, causing severe undersupply and >100% price spikes for NOR flash and SLC NAND; Morgan Stanley/JPMorgan/TrendForce warn shortages into 2026+ (Source 1). https://www.tomshardware.com/pc-components/dram/nor-flash-and-slc-nand-production-are-under-threat-as-capacity-gets-routed-to-more-profitable-products-severe-undersupply-threatens-everyday-electronics\n\n- MLCC reallocation: Makers are shifting lines from mainstream MLCCs to higher‑end AI/server grades, Murata is discontinuing parts, prices and long‑term contracts are rising (Source 2). https://www.koreaherald.com/article/10877230\n\n- Geopolitical/logistics shifts: Some firms that left China are returning because other hubs lack China’s supplier ecosystem and reliable power (Source 3). https://international.astroawani.com/global-news/companies-left-china-dodge-tariffs-now-some-are-headin

## 5. Corrective RAG: check before answering

Retrieval can return a relevant-looking but outdated passage. Corrective RAG adds a small review step:

`retrieve -> grade the evidence -> answer or retry`

Our documents intentionally contain an old 12-month warranty reference and the current 24-month policy. The evaluator should prefer the controlled, newer policy.

In [11]:
def corrective_rag(question):
    # Start with one deliberately outdated passage to simulate poor retrieval.
    first_hits = [
        item for item in records
        if item["source"] == "04_q3_operations_report.pdf" and item["page"] == 7
    ]
    first_context = "\n\n".join(
        f"[{h['source']}, page {h['page']}] {h['text']}" for h in first_hits
    )

    grade = openai_client.responses.create(
        model=CHAT_MODEL,
        instructions="Grade the evidence for answering the question. First line must be GOOD or RETRY. Prefer current controlled policy over historical or superseded values. Then give one short reason.",
        input=f"Question: {question}\n\nEvidence:\n{first_context}",
    ).output_text.strip()
    print("Evidence grade:\n", grade)

    hits = first_hits
    if grade.upper().startswith("RETRY"):
        print("\nRetrying with a current-policy query...")
        hits = retrieve("current effective Northstar-branded device warranty policy version 3.2", top_k=4)

    context = "\n\n".join(
        f"[{h['source']}, page {h['page']}] {h['text']}" for h in hits
    )
    answer = openai_client.responses.create(
        model=CHAT_MODEL,
        instructions="Answer only from the evidence. Cite filename and page. Resolve conflicts using version and effective date.",
        input=f"Question: {question}\n\nEvidence:\n{context}",
    ).output_text
    print("\nFinal answer:\n", answer)
    return answer

corrective_rag("What is the current warranty period for Northstar-branded devices?")

Evidence grade:
 RETRY
The excerpt only notes that the 12-month value was superseded by policy v3.2 (effective 1 July 2026) but does not state the current warranty period under the controlled policy.

Retrying with a current-policy query...

Final answer:
 The current warranty period is 24 months (a 24-month limited hardware warranty) for Northstar‑branded devices purchased on or after 1 July 2026 (CSP‑104, version 3.2). (01_customer_returns_policy.pdf, p.1; 01_customer_returns_policy.pdf, p.5; 04_q3_operations_report.pdf, p.4).


'The current warranty period is 24 months (a 24-month limited hardware warranty) for Northstar‑branded devices purchased on or after 1 July 2026 (CSP‑104, version 3.2). (01_customer_returns_policy.pdf, p.1; 01_customer_returns_policy.pdf, p.5; 04_q3_operations_report.pdf, p.4).'

In [12]:
def choose_route(question):
    text = question.lower()
    if any(word in text for word in ["latest", "recent", "today", "this week", "web"]):
        return "WEB", "The question needs fresh external information."
    if any(word in text for word in ["total", "revenue", "average", "q2", "q3", "largest increase"]):
        return "SQL", "The question requires an exact calculation."
    if any(phrase in text for phrase in ["connected", "how is", "relationship", "depends on"]):
        return "GRAPH", "The question asks about relationships across entities."
    return "DOCUMENTS", "The answer is likely contained in policy or report text."

questions = [
    "What is the damaged-product return window?",
    "How is Helios Components connected to Project Atlas?",
    "Which region had the largest increase from Q2 to Q3?",
    "What recent events are affecting electronics supply chains?",
]

for question in questions:
    route, reason = choose_route(question)
    print(f"{route:10} | {question}\n           {reason}\n")

DOCUMENTS  | What is the damaged-product return window?
           The answer is likely contained in policy or report text.

GRAPH      | How is Helios Components connected to Project Atlas?
           The question asks about relationships across entities.

SQL        | Which region had the largest increase from Q2 to Q3?
           The question requires an exact calculation.

WEB        | What recent events are affecting electronics supply chains?
           The question needs fresh external information.



## 6. Decision matrix and takeaway

| Approach | Best source | Best question type |
|---|---|---|
| Vector RAG | Policies and reports | “What does this document say?” |
| Graph RAG | Entities and relationships | “How are these things connected?” |
| Text-to-SQL | Tables | “What is the exact total or comparison?” |
| Exa Web RAG | Live public web | “What is happening now?” |
| Vision RAG | Charts and visual layouts | “What does this diagram show?” |
| Corrective RAG | Any retrieved evidence | “Is this context good enough to answer?” |

### The big idea

**Advanced RAG is not about replacing vector search. It is about routing each question to the retrieval engine that matches the shape of the answer.**

Try changing the questions, inspecting the intermediate evidence, and noticing which parts are retrieval and which parts are generation.